In [1]:
!pip install scib-metrics --quiet
import scib_metrics
print(f"scib-metrics: {scib_metrics.__version__}")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
scib-metrics: 0.5.9


In [3]:
import json
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
from sklearn.metrics import silhouette_score
import os

with open("benchmark_config.json") as f:
    config = json.load(f)

# Pipeline cluster files to evaluate
PIPELINES = {
    "scanpy_cpu": ("harmonized", "scanpy_like"),
    "rsc_gpu_0141": ("harmonized", "scanpy_like"),
    "rsc_gpu_015": ("harmonized", "scanpy_like"),
    "seurat_cpu": ("harmonized", "seurat"),
    "scalesc_gpu": ("harmonized", "scalesc"),
}

DATASETS = {
    "pbmc3k": "pbmc3k",
    "lung65k": "lung65k",
}

results = []

for ds_key, config_key in DATASETS.items():
    dcfg = config["datasets"][config_key]
    prefix = dcfg["pipeline_prefix"]
    
    # Load canonical h5ad with PCA embedding
    adata = sc.read_h5ad(dcfg["canonical_h5ad"])
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=10000)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
                                 flavor="seurat_v3", subset=False, inplace=True)
    sc.pp.pca(adata, n_comps=50, svd_solver="arpack", random_state=0, mask_var="highly_variable")
    adata.obs_names = adata.obs_names.str.replace("-1$", "", regex=True)
    
    X_pca = adata.obsm["X_pca"]
    
    print(f"\n{'='*60}")
    print(f"{dcfg['name']} — scib-metrics")
    print(f"{'='*60}")
    
    for pipe_name, (suffix, kind) in PIPELINES.items():
        cluster_file = f"write/{prefix}_{pipe_name}_{suffix}_clusters.csv"
        if not os.path.exists(cluster_file):
            continue
        
        df = pd.read_csv(cluster_file)
        df["barcode"] = df["barcode"].str.replace("-1$", "", regex=True)
        df = df.set_index("barcode")
        
        common = adata.obs_names.intersection(df.index)
        if len(common) < 100:
            continue
        
        labels = df.loc[common, "leiden"].astype(str).values
        idx = np.array([list(adata.obs_names).index(bc) for bc in common])
        X = X_pca[idx]
        
        n_clusters = len(set(labels))
        
        # 1. Silhouette (ASW) — cluster compactness/separation on shared PCA
        asw = silhouette_score(X, labels) if n_clusters > 1 else np.nan
        
        # 2. Calinski-Harabasz — ratio of between-cluster to within-cluster variance
        from sklearn.metrics import calinski_harabasz_score
        ch = calinski_harabasz_score(X, labels) if n_clusters > 1 else np.nan
        
        # 3. Davies-Bouldin — lower is better, cluster similarity measure
        from sklearn.metrics import davies_bouldin_score
        db = davies_bouldin_score(X, labels) if n_clusters > 1 else np.nan
        
        # 4. scib-metrics silhouette per label
        try:
            asw_label = float(scib_metrics.silhouette_label(X, labels))
        except Exception:
            asw_label = np.nan
        
        results.append({
            "dataset": dcfg["name"],
            "pipeline": pipe_name,
            "n_clusters": n_clusters,
            "ASW": round(asw, 4),
            "ASW_label": round(asw_label, 4) if not np.isnan(asw_label) else "-",
            "Calinski_Harabasz": round(ch, 1),
            "Davies_Bouldin": round(db, 4),
        })
        
        print(f"  {pipe_name:20s}: {n_clusters:2d} cl | ASW={asw:.4f} | CH={ch:.1f} | DB={db:.4f}")
    
    del adata

# Summary table
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
results_df.to_csv("write/scib_metrics_summary.csv", index=False)
print("\nSaved: write/scib_metrics_summary.csv")


PBMC3k — scib-metrics


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


  scanpy_cpu          :  9 cl | ASW=0.1135 | CH=431.4 | DB=2.2219
  rsc_gpu_0141        :  9 cl | ASW=0.1106 | CH=429.2 | DB=2.2493
  rsc_gpu_015         :  9 cl | ASW=0.1106 | CH=429.2 | DB=2.2493
  seurat_cpu          : 10 cl | ASW=0.1179 | CH=372.0 | DB=2.4225
  scalesc_gpu         :  9 cl | ASW=0.1180 | CH=417.1 | DB=2.1237

Human Lung Cell 65K — scib-metrics
  scanpy_cpu          : 38 cl | ASW=0.2192 | CH=9663.2 | DB=1.5772
  rsc_gpu_0141        : 37 cl | ASW=0.2070 | CH=9602.5 | DB=1.5989
  rsc_gpu_015         : 38 cl | ASW=0.2080 | CH=9524.4 | DB=1.5936
  seurat_cpu          : 44 cl | ASW=0.1690 | CH=8395.6 | DB=1.5800
  scalesc_gpu         : 38 cl | ASW=0.1863 | CH=9405.2 | DB=1.7605

SUMMARY
            dataset     pipeline  n_clusters    ASW  ASW_label  Calinski_Harabasz  Davies_Bouldin
             PBMC3k   scanpy_cpu           9 0.1135     0.5568              431.4          2.2219
             PBMC3k rsc_gpu_0141           9 0.1106     0.5553              429.2          2.2

In [2]:
import json, os
import numpy as np
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc
import scib_metrics
from scib_metrics.nearest_neighbors import NeighborsResults
from sklearn.neighbors import NearestNeighbors
import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(pool_allocator=True, managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)

with open("benchmark_config.json") as f:
    cfg = json.load(f)
prefix = cfg["datasets"]["lung65k"]["pipeline_prefix"]

# Load and prep on GPU
adata = sc.read_h5ad("write/lung_65k_canonical_filtered.h5ad")
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
sc.pp.highly_variable_genes(adata, layer="counts", n_top_genes=5000, flavor="seurat_v3", subset=False)
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.pca(adata, n_comps=50, mask_var="highly_variable")
rsc.get.anndata_to_CPU(adata)
adata.obs_names = adata.obs_names.str.replace("-1$", "", regex=True)

# Build NeighborsResults via sklearn on PCA
print("Computing kNN for LISI...")
X_pca = adata.obsm["X_pca"]
nn = NearestNeighbors(n_neighbors=15, metric="euclidean", algorithm="brute")
nn.fit(X_pca)
distances, indices = nn.kneighbors(X_pca)
knn = NeighborsResults(indices=indices, distances=distances)
print(f"Done: {indices.shape}")

PIPELINES = {
    "scanpy_cpu": f"write/{prefix}_scanpy_cpu_harmonized_clusters.csv",
    "rsc_gpu_0141": f"write/{prefix}_rsc_gpu_0141_harmonized_clusters.csv",
    "rsc_gpu_015": f"write/{prefix}_rsc_gpu_015_harmonized_clusters.csv",
    "seurat_cpu": f"write/{prefix}_seurat_cpu_harmonized_clusters.csv",
    "scalesc_gpu": f"write/{prefix}_scalesc_gpu_harmonized_clusters.csv",
}

for pipe_name, cluster_file in PIPELINES.items():
    if not os.path.exists(cluster_file):
        continue
    df = pd.read_csv(cluster_file)
    df["barcode"] = df["barcode"].str.replace("-1$", "", regex=True)
    common = adata.obs_names.intersection(df.set_index("barcode").index)
    idx = np.array([list(adata.obs_names).index(bc) for bc in common])

    knn_sub = NeighborsResults(indices=indices[idx], distances=distances[idx])
    labels = np.array(df.set_index("barcode").loc[common, "leiden"].values, dtype=str)
    batch = np.array(adata.obs["patient"].iloc[idx].values, dtype=str)

    try:
        cilisi = float(scib_metrics.clisi_knn(knn_sub, labels))
    except Exception as e:
        cilisi = f"ERR: {e}"

    try:
        ilisi = float(scib_metrics.ilisi_knn(knn_sub, batch))
    except Exception as e:
        ilisi = f"ERR: {e}"

    print(f"  {pipe_name:20s}: CiLISI={cilisi}  iLISI={ilisi}")

del adata

/home/sjeong7/.local/lib/python3.13/site-packages/rapids_singlecell/__init__.py:38: UserWarning: 
Multiple rapids_singlecell packages are installed: rapids-singlecell, rapids-singlecell-cu12
Please uninstall all versions and reinstall only one:
  pip uninstall rapids-singlecell rapids-singlecell-cu12

  _detect_duplicate_installation()


Computing kNN for LISI...
Done: (65462, 15)


2026-04-03 10:43:27 | [INFO] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-04-03 10:43:27 | [WARNING] An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


  scanpy_cpu          : CiLISI=1.0  iLISI=5.960464477539063e-08
  rsc_gpu_0141        : CiLISI=1.0  iLISI=5.960464477539063e-08
  rsc_gpu_015         : CiLISI=1.0  iLISI=5.960464477539063e-08
  seurat_cpu          : CiLISI=1.0  iLISI=5.960464477539063e-08
  scalesc_gpu         : CiLISI=1.0  iLISI=5.960464477539063e-08
